# 05. The agent, one node at a time

**Goal:** see what each LangGraph node does to the state, then watch the human-review interrupt pause
a run and resume it from the SQLite checkpoint.

This notebook runs in **offline mode** (`offline=True`): the Claude nodes are replaced by fixed stubs
that use the reference graph, so it needs no API key. The tool nodes are the real code. To keep it
quick the refuters use 10 half-samples here.

## Setup

Imports and configuration. Every threshold and setting comes from `config.yaml`. Set `SYNTHETIC = True` to run on the synthetic file built by `python -m causallab prepare --synthetic`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from causallab.config import load_config, load_spec, results_dir

cfg = load_config()
SYNTHETIC = False   # True = run on the synthetic Expedia-like file (no Kaggle data needed)
pd.set_option("display.width", 140)
cfg["refuters"]["subset_n"] = 10
from causallab.agent import nodes as N
from causallab.agent.state import AgentState
rt = N.make_runtime("walkthrough", cfg)
rt.trace_path.unlink(missing_ok=True)  # a re-run starts a fresh trace instead of appending to the last one
state = AgentState(run_id="walkthrough", question="What is the causal effect of a top-3 position on clicks?",
                   offline=True, synthetic=SYNTHETIC, auto_approve=True)

def apply(state, updates):
    """What LangGraph does between nodes: merge the node's updates into the state."""
    return state.model_copy(update=updates)

## intake

**What:** turns the question into a typed spec: treatment, outcome, unit, population, target arm.

**Why:** everything downstream reads these typed fields. Code then enforces the firewall: a top-3
question can only target the observational arm.

**Expect:** `top3`, `click_bool`, `observational`.

In [2]:
state = apply(state, N.intake(rt, state)); state.spec

QuestionSpec(treatment='top3', outcome='click_bool', unit='one hotel shown in one search', population='algorithm-ranked searches, positions 1-10 without reserved slots', target_arm='observational')

## profile_data

**What:** rows, searches, treatment share, outcome rate, most-missing columns, from the target arm only.

In [3]:
state = apply(state, N.profile_data(rt, state)); state.profile

{'rows': 1218303,
 'groups': 140424,
 'arm': 'observational',
 'treatment_share': 0.3455987549895223,
 'outcome_rate': 0.09439687828069043,
 'columns': 57,
 'most_missing': {'visitor_hist_starrating': 0.9372627334907654,
  'visitor_hist_adr_usd': 0.9370214142130489,
  'srch_query_affinity_score': 0.9252016944881528,
  'gross_bookings_usd': 0.9179013759302899,
  'log_distance': 0.3268702449226506,
  'prop_log_historical_price': 0.1453497200614297,
  'prop_location_score2': 0.13794679976984378,
  'loc2_rel': 0.13794679976984378}}

## propose_dag

**What:** each variable's role, an explicit list of directed edges, the adjustment set, the excluded
variables with reasons, and unobserved threats. Claude sees variable **descriptions only**, never data.

**Why the explicit edges:** the identification check builds the graph from the edges. Built from role
labels alone, every 'confounder' would be wired to treatment and outcome and the check would always pass.

**Expect:** 44 adjustment variables; `booking_bool` and `position` excluded.

In [4]:
state = apply(state, N.propose_dag(rt, state))
print(len(state.dag.adjustment_set), "adjustment variables;", len(state.dag.edges), "edges")
[(x.name, x.reason) for x in state.dag.excluded]

44 adjustment variables; 92 edges


[('gross_bookings_usd', 'post-outcome'),
 ('position', 'defines the treatment'),
 ('booking_bool', 'downstream of the click')]

## human_review

**What:** in a real run this node pauses and waits for a person (shown at the end of the notebook).
Here `auto_approve=True`, so it approves at once.

**Why:** approval sits on the **assumptions**, before any estimate exists, so nobody approves a graph
because they liked the number it produced.

In [5]:
state = apply(state, N.human_review(rt, state)); state.review

Review(decision='approve', adjustment_set=None, notes='')

## identify

**What:** the back-door criterion on the approved graph: (1) nothing in the adjustment set is a
descendant of the treatment, and (2) the set blocks every back-door path (DoWhy's d-separation check).

**Why both:** DoWhy's function covers (2) only; on its own it would accept adjusting for a mediator.

**Expect:** valid.

In [6]:
state = apply(state, N.identify(rt, state)); state.identification

{'valid': True,
 'problems': [],
 'd_separated': True,
 'descendants_in_set': [],
 'merged_into_treatment': ['position'],
 'n_nodes': 48,
 'n_edges': 91}

## route_design, estimate, refute

**What:** check the treatment varies; run the five estimators; run the checks. `estimate` also saves the
propensity model, and the **benchmark module** (not this path) uses it to trim the randomly ranked arm
and compute the trimmed benchmark and the gap.

**Expect:** a few minutes; the AIPW estimate and a gap verdict.

In [7]:
state = apply(state, N.route_design(rt, state))
state = apply(state, N.estimate(rt, state))
print({k: round(v["estimate"] * 100, 2) for k, v in state.estimates.items()})
print("gap:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in state.gap["primary"].items()})

{'naive_untrimmed': 9.77, 'naive': 9.77, 'regression': 8.04, 'ipw': 6.96, 'aipw': 6.61, 'dml': 6.53}
gap: {'gap': 0.0136, 'se': 0.001, 'ci_low': 0.0116, 'ci_high': 0.0157, 'tolerance': 0.0105, 'verdict': 'mismatch'}


In [8]:
state = apply(state, N.refute(rt, state))
{k: v["passed"] for k, v in state.refutations.items()}

{'overlap': True,
 'balance': True,
 'placebo': True,
 'random_common_cause': True,
 'subset_stability': True,
 'sensitivity': True,
 'held_out_covariate': None}

## judge

**What:** code applies the thresholds and picks the route: pass, re-specify (balance failed, at most
twice, never dropping an approved confounder) or refuse (overlap, placebo, random common cause, subset
stability or sensitivity failed, or balance still failed after two re-specifications).
Claude only writes the explanation.

**Expect:** `pass` (every check passed).

In [9]:
state = apply(state, N.judge(rt, state)); state.verdict

{'route': 'pass',
 'failures': [],
 'rationale': 'Every pass-or-fail check passed, so the estimate is reported.'}

## narrate

**What:** the report. Claude may only write prose with named placeholders such as `{primary_estimate}`;
code fills them from state, and a guard rejects the report if any number in it was not written by code.

In [10]:
state = apply(state, N.narrate(rt, state))
print((rt.run_dir / "report.md").read_text(encoding="utf-8")[:2500])

# CausalLab report: What is the causal effect of a top-3 position on clicks?

Run `walkthrough` · status: **reported** · dataset: expedia

## Question

- Treatment: `top3` · Outcome: `click_bool` · Target arm: observational
- Unit: one hotel shown in one search
- Population: algorithm-ranked searches, positions 1-10 without reserved slots

## Summary

The primary doubly robust estimate of the effect is 6.61 percentage points, with a ninety-five percent interval from 6.49 to 6.73. The naive comparison gave 9.77 percentage points. The randomly ranked benchmark, trimmed by the same rule, is 5.25 percentage points; the gap is 1.36 percentage points (mismatch).

## Estimates (percentage points, 95% interval, clustered by `srch_id`)

| Estimator | Effect (pp) | 95% interval | Gap to benchmark (pp) |
|---|---|---|---|
| Experimental benchmark (trimmed) | 5.25 | [5.09, 5.42] | n/a |
| Naive, before trimming | 9.77 | [9.64, 9.89] | 4.52 [4.31, 4.72], mismatch |
| Naive difference in means | 9.7

## The interrupt and resume, explicitly

**What:** the compiled graph with a SQLite checkpointer. Without `auto_approve`, the run stops at
`human_review` and returns the review request. Calling `resume` with a decision continues from the
checkpoint, even in a new Python process.

**Why:** a person must be able to read the assumptions, take their time, and come back.

This demo uses a small synthetic scenario so it runs in seconds.

In [11]:
from causallab.agent import graph
r = graph.start("Does the featured slot cause more clicks?", dataset="trap:clean_confounding", offline=True, cfg=cfg)
print("paused:", r["paused"])
r["review_request"]

paused: True


{'run_id': 'run-e0e576fb',
 'attempt': 0,
 'treatment': 'featured_slot',
 'outcome': 'clicked',
 'target_arm': 'observational',
 'adjustment_set': ['listing_quality',
  'listing_price',
  'review_count',
  'photo_count',
  'host_response_rate',
  'user_tenure_days'],
 'excluded': [],
 'unobserved_threats': [],
 'trim': None,
 'interactions': [],
 'n_edges': 13}

In [12]:
r = graph.resume(r["run_id"], {"decision": "approve"}, cfg)
print("paused:", r["paused"], "| status:", r["state"]["outcome_status"], "| route:", r["state"]["verdict"]["route"])

paused: False | status: reported | route: pass


## The graph diagram

In [13]:
# its own file: `python -m causallab diagram` writes the published results/agent_graph.mmd
print(graph.export_diagram(results_dir(cfg) / "agent_graph_notebook.mmd", cfg).read_text(encoding="utf-8"))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	intake(intake)
	profile_data(profile_data)
	propose_dag(propose_dag)
	human_review(human_review)
	identify(identify)
	route_design(route_design)
	estimate(estimate)
	refute(refute)
	judge(judge)
	heterogeneity(heterogeneity)
	narrate(narrate)
	refuse(refuse)
	__end__([<p>__end__</p>]):::last
	__start__ --> intake;
	estimate --> refute;
	heterogeneity --> narrate;
	human_review -.-> identify;
	human_review -.-> refuse;
	identify -.-> refuse;
	identify -.-> route_design;
	intake --> profile_data;
	judge -.-> heterogeneity;
	judge -.-> narrate;
	judge -.-> propose_dag;
	judge -.-> refuse;
	profile_data --> propose_dag;
	propose_dag --> human_review;
	refute --> judge;
	route_design -.-> estimate;
	route_design -.-> refuse;
	narrate --> __end__;
	refuse --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

